In [ ]:
import os
import re
from typing import Any, Dict

from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')
DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

client = MongoClient(MONGODB_URI)
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [ ]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)
retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store,
    search_index_name="search_index",
    top_k=5,
    fulltext_penalty=50,
    vector_penalty=50,
    post_filter=[
        {
            '$project': {
                'embedding': 0,
            }
        }
    ]
)

In [ ]:
def build_pre_filter(where: Dict[str, Dict[str, Any]] | None = None) -> Dict[str, Any]:
    ALLOWED_FIELDS = {"article", "anchor_page"}
    ALLOWED_OPS = {"$eq", "$ne", "$in", "$nin", "$gt", "$gte", "$lt", "$lte"}
    
    if not where:
        return {}
    clauses = []
    for field, ops in where.items():
        if field not in ALLOWED_FIELDS:
            raise ValueError(f"不支援欄位: {field}")
        if not isinstance(ops, dict):
            raise ValueError(f"{field} 的條件必須是 dict")
        field_ops = {}
        for op, value in ops.items():
            if op not in ALLOWED_OPS:
                raise ValueError(f"{field} 不支援運算子: {op}")
            field_ops[op] = value
        clauses.append({field: field_ops})
    if len(clauses) == 1:
        return clauses[0]
    return {"$and": clauses}